[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_54_Developer_Experience.ipynb)

# Lesson 54 -- Phase 5 - Developer Experience
## Docs, Tests, Contribution Guide & Tooling

**Phase 5 roadmap:**

| Lesson | Topic | Status |
|--------|-------|--------|
| L51 | Architecture & Scaffold | done |
| L52 | Advanced Retrieval | done |
| L53 | Production Deployment | done |
| **L54** | **Developer Experience -- you are here** | in progress |
| L55 | Capstone -- Ship It | next |

An open-source project lives or dies on its **developer experience (DevEx)**.
Great code nobody can run, test, or contribute to won't gain traction.

**This lesson covers four pillars:**

1. **Documentation** -- mkdocs-material + auto-generated API reference
2. **Testing** -- pytest fixtures, parametrize, async tests, conftest.py
3. **Coverage Gate** -- enforcing minimum test coverage in CI
4. **Tooling** -- pre-commit hooks (ruff/mypy), CONTRIBUTING.md, SDK polish

Everything writes files into `/content/auto_researcher_v2/` from L51.


In [ ]:
# @title Setup -- Install dependencies & load API key
!pip install anthropic mkdocs mkdocs-material "mkdocstrings[python]" \
    pytest pytest-cov pytest-asyncio coverage ruff \
    nest_asyncio pydantic pydantic-settings pyyaml -q

import os, sys, json, textwrap, subprocess
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise EnvironmentError("Set ANTHROPIC_API_KEY as a Colab Secret or env var")
    print("API key loaded from environment")

PROJECT_ROOT = Path("/content/auto_researcher_v2")
print(f"Project root: {PROJECT_ROOT}")


## 1. Why Developer Experience Matters

The graveyard of open-source AI projects is filled with repos that had:

- Great ideas but no documentation -> nobody could understand it
- Zero tests -> every PR broke something -> contributors gave up
- No contribution guide -> PRs arrived in random formats -> maintainer burned out
- No linting -> inconsistent style -> code reviews became style wars

**The rule:** A contributor should go from `git clone` to a passing test run in **under 5 minutes**.

### The Four Pillars

```
+------------------+---------------------------+---------------------+
|  Pillar          |  Tool                     |  Output             |
+------------------+---------------------------+---------------------+
| Documentation    |  mkdocs-material          |  /docs site         |
|                  |  mkdocstrings             |  API reference      |
+------------------+---------------------------+---------------------+
| Testing          |  pytest + pytest-asyncio  |  test suite         |
|                  |  conftest.py fixtures     |  reusable mocks     |
+------------------+---------------------------+---------------------+
| Coverage         |  pytest-cov + .coveragerc |  coverage report    |
|                  |  CI gate (sys.exit)       |  badge + gate       |
+------------------+---------------------------+---------------------+
| Tooling          |  ruff + mypy              |  .pre-commit config |
|                  |  CONTRIBUTING.md          |  contribution guide |
+------------------+---------------------------+---------------------+
```


In [ ]:
# Scaffold check / creation
# If L51 was already run, auto_researcher_v2 exists. Otherwise we create minimal structure.

DIRS = [
    "auto_researcher", "auto_researcher/reliability",
    "auto_researcher/inference", "auto_researcher/retrieval",
    "tests", "evals", "docs", "docs/api",
    ".github/ISSUE_TEMPLATE", ".github",
]
for d in DIRS:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# Create pipeline.py with proper Google-style docstrings
pipeline_src = (
    '"""Research pipeline -- plan, search, draft, critique."""\n'
    "from __future__ import annotations\n"
    "from dataclasses import dataclass\n"
    "from typing import Protocol, runtime_checkable\n\n\n"
    "@dataclass(frozen=True)\n"
    "class ResearchResult:\n"
    '    """Immutable result returned by the research pipeline.\n\n'
    "    Attributes:\n"
    "        query: The original research question.\n"
    "        answer: Synthesised answer with inline citations.\n"
    "        cost_usd: Total API spend for this request.\n"
    "        latency_s: Wall-clock time in seconds.\n"
    '    """\n'
    "    query: str\n"
    "    answer: str\n"
    "    cost_usd: float\n"
    "    latency_s: float\n\n\n"
    "@runtime_checkable\n"
    "class InferenceBackend(Protocol):\n"
    '    """Pluggable LLM backend (Anthropic, vLLM, mock).\n\n'
    "    Any callable class that exposes ``complete`` and ``cost_usd``\n"
    "    satisfies this protocol -- no inheritance required.\n"
    '    """\n\n'
    "    async def complete(self, system: str, user: str) -> str:\n"
    '        """Return model text response.\n\n'
    "        Args:\n"
    "            system: System-prompt text.\n"
    "            user: User-turn text.\n\n"
    "        Returns:\n"
    "            Model response as a plain string.\n"
    '        """\n'
    "        ...\n\n"
    "    def cost_usd(self, input_tokens: int, output_tokens: int) -> float:\n"
    '        """Compute cost for a given token pair.\n\n'
    "        Args:\n"
    "            input_tokens: Tokens in the prompt.\n"
    "            output_tokens: Tokens in the completion.\n\n"
    "        Returns:\n"
    "            Cost in US dollars.\n"
    '        """\n'
    "        ...\n"
)
(PROJECT_ROOT / "auto_researcher" / "pipeline.py").write_text(pipeline_src)

result = subprocess.run(
    ["find", str(PROJECT_ROOT), "-type", "f", "-name", "*.py"],
    capture_output=True, text=True)
for line in sorted(result.stdout.strip().split("\n"))[:12]:
    print(line.replace(str(PROJECT_ROOT) + "/", ""))
print("\nScaffold ready")


## 2. Documentation with mkdocs-material

**mkdocs-material** generates a beautiful static site from Markdown files.
**mkdocstrings** auto-generates API reference pages from Python docstrings.

### Directory layout

```
auto_researcher_v2/
├── docs/
│   ├── index.md          <- home page
│   ├── quickstart.md     <- 5-minute getting-started guide
│   ├── concepts.md       <- architectural overview
│   └── api/
│       └── pipeline.md   <- ::: auto_researcher.pipeline  (auto-generated)
└── mkdocs.yml            <- site configuration
```

### The magic directive

```markdown
# Pipeline API

::: auto_researcher.pipeline.ResearchResult
    options:
      show_source: true
      heading_level: 3
```

That single directive reads the Python docstring and renders full API docs.
Change the docstring -> rebuild -> docs stay in sync automatically.


In [ ]:
import yaml

# mkdocs.yml
mkdocs_config = {
    "site_name": "auto-researcher",
    "site_description": "Open-source AI research assistant",
    "repo_url": "https://github.com/your-org/auto-researcher",
    "repo_name": "your-org/auto-researcher",
    "theme": {
        "name": "material",
        "features": ["navigation.tabs", "navigation.top", "search.highlight", "content.code.copy"],
    },
    "plugins": [
        "search",
        {"mkdocstrings": {
            "handlers": {
                "python": {
                    "options": {
                        "show_source": True,
                        "show_root_heading": True,
                        "docstring_style": "google",
                    }
                }
            }
        }}
    ],
    "nav": [
        {"Home": "index.md"},
        {"Quickstart": "quickstart.md"},
        {"Concepts": "concepts.md"},
        {"API Reference": [{"Pipeline": "api/pipeline.md"}]},
    ],
}
with open(PROJECT_ROOT / "mkdocs.yml", "w") as f:
    yaml.dump(mkdocs_config, f, default_flow_style=False)
print("mkdocs.yml written")

# docs/index.md
(PROJECT_ROOT / "docs" / "index.md").write_text(
    "# auto-researcher\n\n"
    "Open-source AI research assistant: plan, search, draft, cite.\n\n"
    "## Quick example\n\n"
    "```python\n"
    "import asyncio\n"
    "from auto_researcher import AutoResearcherV2, AutoResearcherConfig\n\n"
    "result = asyncio.run(\n"
    "    AutoResearcherV2(AutoResearcherConfig()).research(\"What is RAG?\")\n"
    ")\n"
    "print(result.answer)\n"
    "```\n"
)

(PROJECT_ROOT / "docs" / "quickstart.md").write_text(
    "# Quickstart\n\n"
    "```bash\npip install auto-researcher\n```\n\n"
    "```bash\nexport ANTHROPIC_API_KEY=sk-ant-...\n```\n\n"
    "```bash\nauto-researcher research \"What is RAG?\"\n```\n"
)

(PROJECT_ROOT / "docs" / "concepts.md").write_text(
    "# Architecture\n\n"
    "AutoResearcherV2 runs: plan -> search -> draft -> critique\n\n"
    "## InferenceBackend Protocol\n\n"
    "Any class implementing `complete(system, user) -> str` qualifies.\n"
    "Swap to a mock for tests with zero friction.\n"
)

(PROJECT_ROOT / "docs" / "api" / "pipeline.md").write_text(
    "# Pipeline API\n\n"
    "::: auto_researcher.pipeline.ResearchResult\n"
    "    options:\n"
    "      show_source: true\n"
    "      heading_level: 3\n\n"
    "::: auto_researcher.pipeline.InferenceBackend\n"
    "    options:\n"
    "      show_source: true\n"
    "      heading_level: 3\n"
)

print("docs/ pages written")
print("\nTo build locally:")
print("  cd /content/auto_researcher_v2 && mkdocs build")
print("  mkdocs serve  # live at http://127.0.0.1:8000")

with open(PROJECT_ROOT / "mkdocs.yml") as f:
    cfg = yaml.safe_load(f)
print(f"\nmkdocs.yml valid -- site_name: {cfg['site_name']!r}")


## 3. Pytest Fixtures & conftest.py

**Fixtures** are pytest's answer to test setup/teardown -- functions that produce
pre-configured objects your tests can use without repetition.

### Why conftest.py?

`conftest.py` at the project root is auto-discovered by pytest.
Fixtures defined there are available to **all** test files -- no imports needed.

### Fixture scopes

| Scope | Created once per | Best for |
|-------|-----------------|----------|
| `function` (default) | test function | mutable state |
| `class` | test class | shared expensive setup |
| `module` | test file | read-only shared resources |
| `session` | entire test run | one-time global setup |

### Fixture composition

```python
@pytest.fixture(scope="session")
def mock_backend():
    return MockInferenceBackend()     # created once for the whole run

@pytest.fixture
def agent(mock_backend):             # receives mock_backend automatically
    return AutoResearcherV2(config, backend=mock_backend)
```

### @pytest.mark.parametrize

```python
@pytest.mark.parametrize("cost,latency", [
    (0.001, 0.5),
    (0.010, 2.3),
])
def test_result(cost, latency):
    r = ResearchResult("q", "a", cost, latency)
    assert r.cost_usd == pytest.approx(cost)
```

This generates **two separate test cases** from one function.


In [ ]:
# Write conftest.py and test files

CONFTEST = (
    '"""Shared pytest fixtures for auto_researcher_v2 tests."""\n'
    "from __future__ import annotations\n"
    "import pytest\n"
    "from dataclasses import dataclass, field\n"
    "\n"
    "\n"
    "@dataclass\n"
    "class MockInferenceBackend:\n"
    '    """Deterministic fake backend -- zero API cost.\n\n'
    "    Attributes:\n"
    "        responses: Mapping of keyword to canned reply.\n"
    "        call_log: Every (system, user) pair seen so far.\n"
    '    """\n'
    "    responses: dict = field(default_factory=lambda: {\n"
    "        \"plan\":    \"1. What is X?\\n2. How does X work?\",\n"
    "        \"search\":  \"[doc-1] X is a technique that ...\",\n"
    "        \"draft\":   \"X is important because [doc-1] ...\",\n"
    "        \"critique\":\"score: 0.85\",\n"
    "    })\n"
    "    call_log: list = field(default_factory=list)\n"
    "\n"
    "    async def complete(self, system: str, user: str) -> str:\n"
    "        self.call_log.append((system, user))\n"
    "        combined = (system + \" \" + user).lower()\n"
    "        for key, reply in self.responses.items():\n"
    "            if key in combined:\n"
    "                return reply\n"
    "        return f\"Mock reply for: {user[:40]}...\"\n"
    "\n"
    "    def cost_usd(self, input_tokens: int, output_tokens: int) -> float:\n"
    "        return 0.0  # free in tests!\n"
    "\n"
    "    def reset(self) -> None:\n"
    "        self.call_log.clear()\n"
    "\n"
    "\n"
    "@pytest.fixture(scope=\"session\")\n"
    "def mock_backend() -> MockInferenceBackend:\n"
    '    """Session-scoped mock backend -- shared across the entire test run."""\n'
    "    return MockInferenceBackend()\n"
    "\n"
    "\n"
    "@pytest.fixture(autouse=False)\n"
    "def reset_backend(mock_backend: MockInferenceBackend):\n"
    '    """Reset call_log before each test that uses mock_backend."""\n'
    "    mock_backend.reset()\n"
    "    yield\n"
)

(PROJECT_ROOT / "tests" / "conftest.py").write_text(CONFTEST)
(PROJECT_ROOT / "tests" / "__init__.py").write_text("")
print("tests/conftest.py written")

TEST_PIPELINE = (
    '"""Tests for core pipeline dataclasses."""\n'
    "import pytest\n"
    "from auto_researcher.pipeline import ResearchResult\n"
    "\n"
    "\n"
    "def test_research_result_fields():\n"
    '    """ResearchResult stores all fields correctly."""\n'
    "    r = ResearchResult(query=\"q\", answer=\"a\", cost_usd=0.001, latency_s=1.2)\n"
    "    assert r.query == \"q\"\n"
    "    assert r.cost_usd == pytest.approx(0.001)\n"
    "\n"
    "\n"
    "def test_research_result_is_immutable():\n"
    '    """ResearchResult is frozen -- cannot be mutated after creation."""\n'
    "    r = ResearchResult(query=\"q\", answer=\"a\", cost_usd=0.0, latency_s=0.5)\n"
    "    with pytest.raises(Exception):\n"
    "        r.query = \"changed\"  # type: ignore[misc]\n"
    "\n"
    "\n"
    "@pytest.mark.parametrize(\"cost,latency\", [\n"
    "    (0.001, 0.5),\n"
    "    (0.010, 2.3),\n"
    "    (0.100, 8.1),\n"
    "])\n"
    "def test_research_result_parametrize(cost, latency):\n"
    '    """Parametrize generates one test case per (cost, latency) pair."""\n'
    "    r = ResearchResult(query=\"q\", answer=\"a\", cost_usd=cost, latency_s=latency)\n"
    "    assert r.cost_usd == pytest.approx(cost)\n"
    "    assert r.latency_s == pytest.approx(latency)\n"
)

(PROJECT_ROOT / "tests" / "test_pipeline.py").write_text(TEST_PIPELINE)
print("tests/test_pipeline.py written")

TEST_MOCK = (
    '"""Tests using the MockInferenceBackend fixture."""\n'
    "import asyncio\n"
    "import pytest\n"
    "from tests.conftest import MockInferenceBackend\n"
    "\n"
    "\n"
    "@pytest.mark.asyncio\n"
    "async def test_mock_returns_plan(mock_backend):\n"
    '    """Backend returns plan reply when 'plan' appears in the prompt."""\n'
    "    mock_backend.reset()\n"
    "    reply = await mock_backend.complete(\"you are a researcher\", \"please plan this\")\n"
    "    assert \"1.\" in reply\n"
    "    assert len(mock_backend.call_log) == 1\n"
    "\n"
    "\n"
    "@pytest.mark.asyncio\n"
    "async def test_mock_fallback(mock_backend):\n"
    '    """Backend returns generic reply for unrecognised input."""\n'
    "    mock_backend.reset()\n"
    "    reply = await mock_backend.complete(\"sys\", \"something totally novel\")\n"
    "    assert \"Mock reply\" in reply\n"
    "\n"
    "\n"
    "def test_mock_cost_is_zero(mock_backend):\n"
    '    """Cost should always be 0.0 for the mock backend."""\n'
    "    assert mock_backend.cost_usd(1_000_000, 500_000) == 0.0\n"
    "\n"
    "\n"
    "@pytest.mark.asyncio\n"
    "async def test_mock_logs_calls(mock_backend):\n"
    '    """Every call is appended to call_log."""\n'
    "    mock_backend.reset()\n"
    "    await mock_backend.complete(\"s1\", \"u1\")\n"
    "    await mock_backend.complete(\"s2\", \"u2\")\n"
    "    assert len(mock_backend.call_log) == 2\n"
    "    assert mock_backend.call_log[0] == (\"s1\", \"u1\")\n"
)

(PROJECT_ROOT / "tests" / "test_mock_backend.py").write_text(TEST_MOCK)
print("tests/test_mock_backend.py written")

print("\n-- Running tests --------------------------------------------------------")
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(PROJECT_ROOT / "tests"),
     "-v", "--tb=short", "--no-header", "--asyncio-mode=auto"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)
out = result.stdout
print(out[-3000:] if len(out) > 3000 else out)
if result.returncode != 0 and result.stderr:
    print("STDERR:", result.stderr[-500:])


## 4. Coverage Gate

**Code coverage** measures what percentage of your source lines are exercised by tests.
A coverage gate in CI prevents merging PRs that drop below a threshold.

### Command

```bash
pytest --cov=auto_researcher --cov-report=term-missing --cov-fail-under=80
```

- `--cov=auto_researcher` -- measure coverage for this package only
- `--cov-report=term-missing` -- show which lines are NOT covered
- `--cov-fail-under=80` -- exit code 1 if coverage < 80%

### .coveragerc

```ini
[run]
source = auto_researcher
omit =
    auto_researcher/cli.py

[report]
exclude_lines =
    pragma: no cover
    def __repr__
    if TYPE_CHECKING:
    raise NotImplementedError
    @abstractmethod
```

### Recommended thresholds

| Coverage | What it means |
|----------|---------------|
| < 60%    | Regressions slip through regularly |
| 60-80%   | Acceptable for prototypes |
| 80-90%   | Good for production libraries |
| > 95%    | Diminishing returns |

Start at 60%, raise 5% per sprint.


In [ ]:
# .coveragerc + coverage CI gate

COVERAGERC = (
    "[run]\n"
    "source = auto_researcher\n"
    "omit =\n"
    "    auto_researcher/__main__.py\n"
    "    auto_researcher/cli.py\n"
    "\n"
    "[report]\n"
    "exclude_lines =\n"
    "    pragma: no cover\n"
    "    def __repr__\n"
    "    if TYPE_CHECKING:\n"
    "    raise NotImplementedError\n"
    "    @abstractmethod\n"
    "    \\.\\.\\.\n"
    "\n"
    "[html]\n"
    "directory = htmlcov\n"
)
(PROJECT_ROOT / ".coveragerc").write_text(COVERAGERC)
print(".coveragerc written")

print("\n-- Running pytest with coverage -----------------------------------------")
cov_result = subprocess.run(
    [sys.executable, "-m", "pytest",
     str(PROJECT_ROOT / "tests"),
     f"--cov={PROJECT_ROOT / 'auto_researcher'}",
     "--cov-report=term-missing",
     "--cov-report=json",
     "--asyncio-mode=auto",
     "-q", "--no-header"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)
)
out = cov_result.stdout
print(out[-3000:] if len(out) > 3000 else out)


def coverage_ci_gate(json_report: Path, min_pct: float = 80.0) -> bool:
    """Read coverage.json and fail if below threshold.

    In CI: replace the print with sys.exit(1) to block the pipeline.

    Args:
        json_report: Path to coverage.json produced by pytest-cov.
        min_pct: Minimum acceptable coverage percentage.

    Returns:
        True if gate passed, False otherwise.
    """
    if not json_report.exists():
        print("coverage.json not found -- skipping gate")
        return True
    with open(json_report) as f:
        data = json.load(f)
    pct = data["totals"]["percent_covered"]
    print(f"\nCoverage: {pct:.1f}%  (threshold: {min_pct}%)")
    if pct < min_pct:
        print(f"FAILED: {pct:.1f}% < {min_pct}%  -- in CI: sys.exit(1)")
        return False
    print(f"PASSED: {pct:.1f}% >= {min_pct}%")
    return True

coverage_ci_gate(PROJECT_ROOT / "coverage.json", min_pct=60.0)

# EXPERIMENT: Raise to 99.0 and see the gate fail.
# In real CI: sys.exit(0 if passed else 1)


## 5. CONTRIBUTING.md

`CONTRIBUTING.md` is the front door for new contributors. GitHub automatically
links to it from the "New Issue" and "New PR" pages.

### Anatomy

```
1. Development setup     <- get running in <5 min
2. Project structure     <- where to find things
3. Making changes        <- branch naming, commit format
4. Running tests         <- exact commands
5. Submitting a PR       <- checklist before opening
6. Code style            <- automated, not manual
7. Documentation         <- how to update docs
8. Getting help          <- Discussions link
```

### Conventional Commits

```
feat(retrieval): add cross-encoder re-ranking
fix(pipeline): handle empty search results gracefully
docs(api): add ResearchResult docstring
test(pipeline): parametrize cost scenarios
```

Tools like `commitizen` and `semantic-release` read these messages to
auto-generate CHANGELOGs and bump version numbers.


In [ ]:
CONTRIBUTING = (
    "# Contributing to auto-researcher\n\n"
    "Thank you! This guide gets you from `git clone` to a passing\n"
    "test run in under 5 minutes.\n\n"
    "---\n\n"
    "## 1. Development setup\n\n"
    "```bash\n"
    "git clone https://github.com/your-org/auto-researcher.git\n"
    "cd auto-researcher\n"
    "python -m venv .venv\n"
    "source .venv/bin/activate\n"
    "pip install -e \".[dev]\"\n"
    "pre-commit install\n"
    "export ANTHROPIC_API_KEY=sk-ant-...\n"
    "```\n\n"
    "---\n\n"
    "## 2. Project structure\n\n"
    "```\n"
    "auto_researcher/       <- source package\n"
    "tests/                 <- pytest suite\n"
    "docs/                  <- mkdocs-material site\n"
    "evals/                 <- golden regression harnesses\n"
    "```\n\n"
    "---\n\n"
    "## 3. Making changes\n\n"
    "1. Create a branch from `main`:\n"
    "   ```bash\n"
    "   git checkout -b feat/your-feature\n"
    "   ```\n"
    "2. Use Conventional Commits:\n"
    "   ```\n"
    "   feat(retrieval): add MMR diversification\n"
    "   fix(pipeline): handle empty search results\n"
    "   ```\n\n"
    "---\n\n"
    "## 4. Running tests\n\n"
    "```bash\n"
    "pytest --cov=auto_researcher --cov-report=term-missing\n"
    "pytest tests/ -m \"not integration\"  # unit tests, no API key needed\n"
    "```\n\n"
    "CI requires >=80% coverage:\n"
    "```bash\n"
    "pytest --cov=auto_researcher --cov-fail-under=80\n"
    "```\n\n"
    "---\n\n"
    "## 5. PR checklist\n\n"
    "- [ ] `pytest` passes locally\n"
    "- [ ] `ruff check auto_researcher/` clean\n"
    "- [ ] `mypy auto_researcher/` clean\n"
    "- [ ] New public functions have Google-style docstrings\n"
    "- [ ] CHANGELOG.md updated under `[Unreleased]`\n\n"
    "---\n\n"
    "## 6. Code style\n\n"
    "```bash\n"
    "ruff check auto_researcher/        # lint\n"
    "ruff format auto_researcher/       # format\n"
    "```\n\n"
    "Pre-commit runs this on every `git commit`.\n\n"
    "---\n\n"
    "## 7. Documentation\n\n"
    "```bash\n"
    "mkdocs serve    # live dev server at http://127.0.0.1:8000\n"
    "mkdocs build    # build static site to site/\n"
    "```\n\n"
    "API docs are auto-generated from docstrings via mkdocstrings.\n\n"
    "---\n\n"
    "## 8. Getting help\n\n"
    "- GitHub Discussions: https://github.com/your-org/auto-researcher/discussions\n"
    "- Issue tracker: https://github.com/your-org/auto-researcher/issues\n"
)

(PROJECT_ROOT / "CONTRIBUTING.md").write_text(CONTRIBUTING)
print(f"CONTRIBUTING.md written ({len(CONTRIBUTING.splitlines())} lines)")

for sec in ["Development setup", "Project structure", "Making changes",
            "Running tests", "PR checklist", "Code style", "Documentation", "Getting help"]:
    assert sec in CONTRIBUTING, f"Missing section: {sec}"
print("All 8 required sections present")


## 6. Pre-commit Hooks

Pre-commit hooks run automatically on `git commit` before the commit lands.
They enforce quality without contributors having to remember to run linters.

### Flow

```
git commit -m "feat: add thing"
  -> pre-commit runs:
    1. ruff check        (lint, auto-fix safe issues)
    2. ruff-format       (format like black)
    3. mypy              (type checking)
    4. trailing-whitespace, end-of-file-fixer
    5. no-commit-to-branch  (blocks direct commits to main)
  -> if any hook fails:
       commit is ABORTED
       auto-fixed files are staged
       you review + re-commit
```

### Ruff replaces three tools

Ruff is written in Rust and replaces flake8 + black + isort.
It is **10-100x faster** and supports the same rule sets.

```toml
[tool.ruff.lint]
select = [
    "E",    # pycodestyle errors
    "F",    # pyflakes (unused imports)
    "I",    # isort (import order)
    "B",    # flake8-bugbear (likely bugs)
    "UP",   # pyupgrade (modernise syntax)
]
```


In [ ]:
PRE_COMMIT_CFG = (
    "# Run: pre-commit install   (one-time setup)\n"
    "# Manual: pre-commit run --all-files\n"
    "repos:\n"
    "  - repo: https://github.com/pre-commit/pre-commit-hooks\n"
    "    rev: v4.6.0\n"
    "    hooks:\n"
    "      - id: trailing-whitespace\n"
    "      - id: end-of-file-fixer\n"
    "      - id: check-yaml\n"
    "      - id: check-toml\n"
    "      - id: check-merge-conflict\n"
    "      - id: debug-statements\n"
    "      - id: no-commit-to-branch\n"
    "        args: [--branch, main]\n"
    "\n"
    "  - repo: https://github.com/astral-sh/ruff-pre-commit\n"
    "    rev: v0.4.9\n"
    "    hooks:\n"
    "      - id: ruff\n"
    "        args: [--fix]\n"
    "      - id: ruff-format\n"
    "\n"
    "  - repo: https://github.com/pre-commit/mirrors-mypy\n"
    "    rev: v1.10.0\n"
    "    hooks:\n"
    "      - id: mypy\n"
    "        additional_dependencies: [pydantic>=2, anthropic]\n"
    "        args: [--ignore-missing-imports]\n"
)
(PROJECT_ROOT / ".pre-commit-config.yaml").write_text(PRE_COMMIT_CFG)
print(".pre-commit-config.yaml written")

PYPROJECT = (
    "[project]\n"
    'name = "auto-researcher"\n'
    'version = "0.2.0"\n'
    'description = "Open-source AI research assistant"\n'
    'requires-python = ">=3.10"\n'
    'license = {text = "MIT"}\n'
    "dependencies = [\n"
    '    "anthropic>=0.25",\n'
    '    "pydantic>=2",\n'
    '    "pydantic-settings>=2",\n'
    "]\n\n"
    "[project.optional-dependencies]\n"
    "dev = [\n"
    '    "pytest>=8",\n'
    '    "pytest-asyncio>=0.23",\n'
    '    "pytest-cov>=5",\n'
    '    "ruff>=0.4",\n'
    '    "mypy>=1.10",\n'
    '    "pre-commit>=3",\n'
    '    "mkdocs-material>=9",\n'
    '    "mkdocstrings[python]>=0.25",\n'
    "]\n\n"
    "[project.scripts]\n"
    'auto-researcher = "auto_researcher.cli:app"\n'
    "\n"
    "[build-system]\n"
    'requires = ["hatchling"]\n'
    'build-backend = "hatchling.build"\n'
    "\n"
    "[tool.ruff]\n"
    "line-length = 100\n"
    'target-version = "py310"\n'
    "\n"
    "[tool.ruff.lint]\n"
    'select = ["E", "W", "F", "I", "B", "UP"]\n'
    'ignore = ["E501"]\n'
    "\n"
    "[tool.mypy]\n"
    'python_version = "3.10"\n'
    "ignore_missing_imports = true\n"
    "disallow_untyped_defs = true\n"
    "\n"
    "[tool.pytest.ini_options]\n"
    'asyncio_mode = "auto"\n'
    'testpaths = ["tests"]\n'
    'addopts = "--tb=short -q"\n'
)
(PROJECT_ROOT / "pyproject.toml").write_text(PYPROJECT)
print("pyproject.toml written")

# Demo ruff on pipeline.py
print("\n-- Ruff lint check -------------------------------------------------------")
r = subprocess.run(
    [sys.executable, "-m", "ruff", "check",
     str(PROJECT_ROOT / "auto_researcher" / "pipeline.py")],
    capture_output=True, text=True
)
print("pipeline.py: no issues" if r.returncode == 0 else r.stdout)


## 7. SDK Polish -- `__all__`, Type Hints, Docstrings, CHANGELOG

### `__all__` -- explicit public API

```python
# Without __all__:
from auto_researcher import *   # imports EVERYTHING including _private things

# With __all__:
__all__ = ["AutoResearcherV2", "AutoResearcherConfig", "ResearchResult"]
# Only those three are exported -- clean, stable public surface
```

### Google-style docstrings (rendered by mkdocstrings)

```python
def research(self, query: str, *, max_cost_usd: float = 1.0) -> ResearchResult:
    """Run the full research pipeline.

    Args:
        query: The research question.
        max_cost_usd: Hard budget cap in USD.

    Returns:
        ResearchResult with answer and cost breakdown.

    Raises:
        BudgetExceeded: If pipeline would exceed max_cost_usd.

    Example:
        >>> result = asyncio.run(agent.research("What is RAG?"))
        >>> print(result.cost_usd)
        0.003
    """
```

### CHANGELOG.md -- Keep a Changelog format

```markdown
## [Unreleased]
### Added
- Cross-encoder re-ranking in retrieval pipeline

## [0.2.0] -- 2026-06-24
### Added
- Production deployment with Docker
```

`semantic-release` reads this format to auto-tag and publish PyPI releases.


In [ ]:
# Polished __init__.py and CHANGELOG.md

INIT_SRC = (
    '"""auto_researcher -- open-source AI research assistant.\n\n'
    "Quickstart::\n\n"
    "    import asyncio\n"
    "    from auto_researcher import AutoResearcherV2, AutoResearcherConfig\n\n"
    "    result = asyncio.run(\n"
    "        AutoResearcherV2(AutoResearcherConfig()).research(\"What is RAG?\")\n"
    "    )\n"
    "    print(result.answer)\n"
    "\n"
    "The library follows semantic versioning.\n"
    '"""\n'
    "from __future__ import annotations\n"
    "\n"
    '__version__: str = "0.2.0"\n'
    '__author__: str = "Gourav Khanijoe"\n'
    '__license__: str = "MIT"\n'
    "\n"
    "from auto_researcher.pipeline import InferenceBackend, ResearchResult\n"
    "\n"
    "__all__: list[str] = [\n"
    '    "ResearchResult",\n'
    '    "InferenceBackend",\n'
    '    "__version__",\n'
    '    "__author__",\n'
    '    "__license__",\n'
    "]\n"
)
(PROJECT_ROOT / "auto_researcher" / "__init__.py").write_text(INIT_SRC)
print("auto_researcher/__init__.py written")

CHANGELOG = (
    "# Changelog\n\n"
    "Format: [Keep a Changelog](https://keepachangelog.com/en/1.1.0/)\n"
    "Versioning: [Semantic Versioning](https://semver.org/)\n\n"
    "---\n\n"
    "## [Unreleased]\n\n"
    "### Added\n"
    "- Developer experience: mkdocs docs site, coverage gate, pre-commit hooks\n"
    "- CONTRIBUTING.md, CHANGELOG.md, SECURITY.md, GitHub health files\n\n"
    "---\n\n"
    "## [0.2.0] -- 2026-06-24\n\n"
    "### Added\n"
    "- Production deployment: Dockerfile, Fly.io + Railway configs\n"
    "- Budget guard with SQLite daily_spend and 402 response\n"
    "- Structured JSON logging with request-ID ContextVar\n"
    "- FastAPI middleware stack: auth, rate limiting, budget guard\n\n"
    "---\n\n"
    "## [0.1.0] -- 2026-06-22\n\n"
    "### Added\n"
    "- Initial 38-file project scaffold\n"
    "- AutoResearcherConfig with pydantic-settings\n"
    "- InferenceBackend Protocol (Anthropic + MockBackend)\n"
    "- AutoResearcherV2 pipeline: plan -> search -> draft -> critique\n"
    "- Typer CLI + FastAPI app factory\n"
    "- GitHub Actions CI with PyPI release trigger\n"
)
(PROJECT_ROOT / "CHANGELOG.md").write_text(CHANGELOG)
print("CHANGELOG.md written")

# Verify __all__ by loading the module
import importlib.util, sys as _sys
_sys.path.insert(0, str(PROJECT_ROOT))
spec = importlib.util.spec_from_file_location(
    "auto_researcher_init",
    str(PROJECT_ROOT / "auto_researcher" / "__init__.py")
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
_sys.path.pop(0)
print(f"\n__all__     = {mod.__all__}")
print(f"__version__ = {mod.__version__}")


## 8. GitHub Community Health Files

GitHub shows a "Community Standards" checklist for every repository.
Having these files unlocks contributor trust and GitHub features:

| File | What it does |
|------|--------------|
| `CONTRIBUTING.md` | Link shown on every new issue/PR form |
| `CODE_OF_CONDUCT.md` | Sets community behaviour expectations |
| `SECURITY.md` | How to report vulnerabilities |
| `.github/ISSUE_TEMPLATE/bug_report.md` | Structured bug reports |
| `.github/ISSUE_TEMPLATE/feature_request.md` | Structured feature requests |
| `.github/pull_request_template.md` | PR checklist for every contributor |
| `.github/CODEOWNERS` | Auto-assigns reviewers by file path |

### CODEOWNERS syntax

```
# Auto-assign reviewers by path
/auto_researcher/inference/   @ml-team
/auto_researcher/retrieval/   @search-team
*                             @gouravkhanijoe   # catch-all
```


In [ ]:
(PROJECT_ROOT / ".github" / "ISSUE_TEMPLATE" / "bug_report.md").write_text(
    "---\n"
    "name: Bug report\n"
    "about: Something is broken\n"
    'title: "[BUG] "\n'
    "labels: bug\n"
    "---\n\n"
    "## Describe the bug\n"
    "A clear description of what went wrong.\n\n"
    "## Steps to reproduce\n"
    "```python\n"
    "from auto_researcher import ...\n"
    "```\n\n"
    "## Expected vs actual behaviour\n"
    "Expected: ...\n"
    "Actual: ...\n\n"
    "## Environment\n"
    "- auto-researcher version:\n"
    "- Python version:\n"
    "- OS:\n"
)
print(".github/ISSUE_TEMPLATE/bug_report.md")

(PROJECT_ROOT / ".github" / "ISSUE_TEMPLATE" / "feature_request.md").write_text(
    "---\n"
    "name: Feature request\n"
    "about: Propose a new capability\n"
    'title: "[FEAT] "\n'
    "labels: enhancement\n"
    "---\n\n"
    "## Problem / motivation\n"
    "What problem does this solve?\n\n"
    "## Proposed solution\n"
    "```python\n"
    "# what you wish you could write\n"
    "```\n\n"
    "## Alternatives considered\n"
    "...\n\n"
    "## Scope\n"
    "- [ ] Breaking change\n"
    "- [ ] New optional dependency\n"
)
print(".github/ISSUE_TEMPLATE/feature_request.md")

(PROJECT_ROOT / ".github" / "pull_request_template.md").write_text(
    "## What does this PR do?\n\n"
    "## Motivation\n"
    "<!-- Closes #123 -->\n\n"
    "## Testing\n"
    "- [ ] `pytest` passes\n"
    "- [ ] `ruff check` clean\n"
    "- [ ] `mypy` clean\n"
    "- [ ] Coverage does not drop below 80%\n"
    "- [ ] CHANGELOG.md updated\n"
)
print(".github/pull_request_template.md")

(PROJECT_ROOT / ".github" / "CODEOWNERS").write_text(
    "# CODEOWNERS -- auto-assigns reviewers by path\n"
    "*                                   @gouravkhanijoe\n"
    "/auto_researcher/inference/         @gouravkhanijoe\n"
    "/auto_researcher/retrieval/         @gouravkhanijoe\n"
    "/auto_researcher/reliability/       @gouravkhanijoe\n"
    "/docs/                              @gouravkhanijoe\n"
    "/.github/                           @gouravkhanijoe\n"
)
print(".github/CODEOWNERS")

(PROJECT_ROOT / "SECURITY.md").write_text(
    "# Security Policy\n\n"
    "## Supported versions\n\n"
    "| Version | Supported |\n"
    "|---------|-----------|\n"
    "| 0.2.x   | yes |\n"
    "| 0.1.x   | no (upgrade to 0.2) |\n\n"
    "## Reporting a vulnerability\n\n"
    "**Do NOT open a public issue for security vulnerabilities.**\n\n"
    "Email: security@example.com\n"
    "We acknowledge within 48 hours and patch critical issues within 14 days.\n"
)
print("SECURITY.md")
print("\nAll GitHub community health files written")


## 9. Ten DevEx Pitfalls

| # | Pitfall | What goes wrong | Fix |
|---|---------|-----------------|-----|
| 1 | **No `__all__`** | `from pkg import *` bleeds internals | Always define `__all__` |
| 2 | **Coverage on wrong path** | `--cov=.` measures test files too -- inflated number | Use `--cov=auto_researcher` |
| 3 | **Coverage gate too high on day 1** | 99% threshold -- nobody passes -- gate disabled | Start at 60%, raise 5% per sprint |
| 4 | **`conftest.py` in `tests/` not root** | Fixtures unavailable to `evals/` tests | Put `conftest.py` at repo root |
| 5 | **Session-scoped fixture with mutable state** | Test A modifies; Test B sees contaminated state | Session = read-only; function = mutable |
| 6 | **Pre-commit on slow operations** | `mypy --strict` takes 30s -- devs bypass with `--no-verify` | Run fast checks on commit; mypy in CI |
| 7 | **mkdocs without `mkdocstrings`** | Docs site exists but API ref is hand-written -- goes stale | Use `:::` directives |
| 8 | **CONTRIBUTING.md never tested** | Guide says `pip install -e ".[dev]"` but dep doesn't exist | CI: fresh checkout + follow guide |
| 9 | **CHANGELOG edited by everyone** | Every merge creates conflicts | Use `[Unreleased]` section + squash-merge |
| 10 | **No `no-commit-to-branch` hook** | Accidental direct push to `main` bypasses CI | Add hook in `.pre-commit-config.yaml` |


In [ ]:
devex_files = [
    "mkdocs.yml",
    "CONTRIBUTING.md",
    "CHANGELOG.md",
    "SECURITY.md",
    ".coveragerc",
    ".pre-commit-config.yaml",
    "pyproject.toml",
    "auto_researcher/__init__.py",
    "auto_researcher/pipeline.py",
    "docs/index.md",
    "docs/quickstart.md",
    "docs/concepts.md",
    "docs/api/pipeline.md",
    "tests/conftest.py",
    "tests/test_pipeline.py",
    "tests/test_mock_backend.py",
    ".github/CODEOWNERS",
    ".github/pull_request_template.md",
    ".github/ISSUE_TEMPLATE/bug_report.md",
    ".github/ISSUE_TEMPLATE/feature_request.md",
]

print("DevEx files written this lesson:\n")
all_ok = True
for f in devex_files:
    path = PROJECT_ROOT / f
    exists = path.exists()
    size = f"{path.stat().st_size:,} B" if exists else "MISSING"
    status = "OK   " if exists else "MISS "
    if not exists:
        all_ok = False
    print(f"  {status} {f:<52}  {size}")

print()
print("All present -- ready for L55" if all_ok else "Some files missing -- re-run cells above")

print("\nLesson 54 summary:")
print("-" * 65)
rows = [
    ("mkdocs.yml + docs/",         "Static site with API reference from docstrings"),
    ("tests/conftest.py",          "Shared fixtures: MockInferenceBackend + reset"),
    ("tests/test_*.py",            "Unit tests: parametrize + async + immutability"),
    (".coveragerc + CI gate",      "Coverage measured, CI fails below threshold"),
    ("CONTRIBUTING.md",            "5-min setup guide + Conventional Commits"),
    (".pre-commit-config.yaml",    "ruff + mypy + no-commit-to-branch hooks"),
    ("pyproject.toml",             "[tool.ruff] + [tool.mypy] + [tool.pytest]"),
    ("__init__.py __all__",        "Explicit public API surface + version string"),
    ("CHANGELOG.md",               "Keep a Changelog format, ready for semver"),
    ("GitHub health files",        "CODEOWNERS + PR template + issue templates"),
]
for component, description in rows:
    print(f"  {component:<35} {description}")


## 10. Homework

1. **Run the full DevEx stack locally** -- clone the repo, follow `CONTRIBUTING.md`
   step by step. Time how long until `pytest` passes. Fix anything that breaks.

2. **Add GitHub Actions docs deployment** -- create `.github/workflows/docs.yml`
   that runs `mkdocs build` on every push to `main` and deploys to GitHub Pages.

3. **Write an integration test** -- add `tests/test_integration.py` with a test
   decorated `@pytest.mark.integration` that actually calls the Anthropic API.
   Gate it: `pytest.importorskip("anthropic")` + skip if no API key set.

4. **Raise the coverage bar** -- write tests until coverage hits 75%,
   then update the CI gate from 60% to 75%.

5. **Commit message automation** -- install `commitizen` (`pip install commitizen`),
   run `cz init`, then use `cz commit` for guided Conventional Commit prompts.

---

## Up Next: Lesson 55 -- Phase 5 Capstone: Ship It

The final lesson puts everything together:

```
L55: Ship It
+-- 1. Integration smoke test (real API, full pipeline)
+-- 2. gh release create v0.2.0 (GitHub release + CHANGELOG)
+-- 3. twine upload to PyPI (publish from CI)
+-- 4. README badges (CI / coverage / PyPI version / license)
+-- 5. Announcement post template
+-- 6. Retrospective: everything built across 55 lessons
```

By the end of L55, `pip install auto-researcher` works from PyPI --
a real, installable, open-source AI agent you built from scratch.
